# MAB2134 — Checkpoint 1: EDA
## Predicting Customer Lifetime Value to Guide Paid Advertising Budget Allocation

**Student:** Christian  
**Course:** MAB2134 Analytics Algorithms 1 (Predictive Analytics 1)  
**Term:** 3rd / SY 2024–2025  
**Instructor:** Millicent H. Singson, MSDS  

---

### Purpose of this notebook
This notebook covers the Data Understanding phase of CRISP-DM for the CLV prediction project. It documents the target distribution, feature distributions, data quality audit, bivariate relationships, and segment analysis. Findings from this EDA directly inform the preprocessing plan and modeling hypotheses submitted at the midterm checkpoint.

---

### Data Provenance

**Source:** Internal company data warehouse (production gold layer). PII scrubbed prior to use.  
**Cohort:** Customers acquired between [START DATE] and [END DATE]. Right-bound selected to ensure all records have a complete 12-month revenue observation window as of the pull date.  
**Pull date:** [DATE OF CSV EXPORT]  
**File used:** `[FILENAME].csv`  

The data was extracted using the following query:

```sql
-- TODO: paste final BQ query here before submission
SELECT
    channel,
    broader_source,
    service_offering,
    is_sales_tql,
    tql_category,
    initial_plan_type,
    max_initial_default_hours,
    max_initial_price_point,
    first_week_mrr,
    company_industry,
    revenue_first_12_months
FROM `[PROJECT].[DATASET].[TABLE]`
WHERE acquisition_date BETWEEN '[START DATE]' AND '[END DATE]'
```

> **Note on reproducibility:** The source data is proprietary company data and cannot be shared externally. The SQL query above documents exactly what was extracted. All analysis in this notebook is reproducible from the CSV file given the query above.

---
## 0. Preliminaries

In [ ]:
# BASIC PACKAGES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# REPRODUCIBILITY
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# DISPLAY OPTIONS
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 100)

# AESTHETICS
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = 'Blues_d'

print('Packages loaded successfully.')

---
## 1. Data Loading & Initial Inspection

In [ ]:
# LOAD DATA
# TODO: update filename
FILE_PATH = '[FILENAME].csv'

df_orig = pd.read_csv(FILE_PATH)

# Always work on a copy; keep original untouched for reference
df = df_orig.copy()

print(f'File loaded: {FILE_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# DTYPES AND NULL SUMMARY
df.info()

In [ ]:
# FIRST ROWS
df.head()

In [ ]:
# DESCRIPTIVE STATISTICS
df.describe(include='all').T

### 1.1 Column Name Cleanup

In [ ]:
# Standardize column names: strip whitespace, lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned column names:')
print(df.columns.tolist())

### 1.2 Summary Snapshot

| Dimension | Value |
|---|---|
| Rows | *(fill after load)* |
| Columns | *(fill after load)* |
| Target | `revenue_first_12_months` |
| Date range | *(fill after load)* |

> **Observations:** *(fill after running the cells above)*

---
## 2. Target Variable Analysis

Understanding the distribution of `revenue_first_12_months` before anything else. This is what the model will predict.

In [ ]:
TARGET = 'revenue_first_12_months'

# Basic stats
print('=== Target Variable Stats ===')
print(df[TARGET].describe())
print(f'\nZero values:    {(df[TARGET] == 0).sum():,}')
print(f'Negative values: {(df[TARGET] < 0).sum():,}')
print(f'Null values:     {df[TARGET].isnull().sum():,}')

In [ ]:
# DISTRIBUTION PLOT — raw and log-transformed side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
sns.histplot(df[TARGET].dropna(), bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Target Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel('12-Month Gross Revenue (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log-transformed (only positive values)
log_target = np.log1p(df[TARGET].dropna())
sns.histplot(log_target, bins=50, ax=axes[1], color='steelblue')
axes[1].set_title('Target Distribution (log1p)', fontweight='bold')
axes[1].set_xlabel('log1p(12-Month Gross Revenue)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Skewness check
print(f'Skewness (raw):      {df[TARGET].skew():.3f}')
print(f'Skewness (log1p):    {log_target.skew():.3f}')

In [ ]:
# PERCENTILE TABLE — useful for understanding value distribution
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_table = df[TARGET].quantile(percentiles).reset_index()
pct_table.columns = ['Percentile', 'Revenue (USD)']
pct_table['Percentile'] = pct_table['Percentile'].apply(lambda x: f'{x:.0%}')
print(pct_table.to_string(index=False))

### 2.1 Target Observations

> *(Fill after running cells above)*
> - Is the target heavily right-skewed? Does log transform help?
> - Are there zero/negative values that need to be investigated or excluded?
> - What does the median vs. mean gap tell us about outliers?

---
## 3. Data Quality Audit

Three checks: missingness, duplicates, and leakage audit.

### 3.1 Missingness

In [ ]:
# Missing value summary, sorted descending
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)

In [ ]:
# Visualize missingness
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.5)))
    sns.barplot(x='Missing %', y=missing_df.index, data=missing_df, color='steelblue', ax=ax)
    ax.set_title('Missingness by Feature (%)', fontweight='bold')
    ax.set_xlabel('% Missing')
    ax.set_ylabel('')
    ax.axvline(40, color='red', linestyle='--', label='40% threshold (company_industry flag)')
    ax.legend()
    plt.tight_layout()
    plt.show()

### 3.2 Duplicates

In [ ]:
# Full-row duplicates
dup_count = df.duplicated().sum()
print(f'Full-row duplicates: {dup_count:,}')

# Note: if there is a customer_id column, also check for duplicate customer IDs
# Uncomment if applicable:
# id_col = 'customer_id'
# print(f'Duplicate {id_col}s: {df[id_col].duplicated().sum():,}')

### 3.3 Leakage Audit

Every feature must be observable **at acquisition time** — before any retention, upsell, or account management action is taken. Features that reveal post-acquisition behavior are leaky.

| Feature | Observable at Acquisition? | Notes |
|---|---|---|
| `channel` | ✅ Yes | Attribution is captured at acquisition |
| `broader_source` | ✅ Yes | Same as above |
| `service_offering` | ✅ Yes | Set at signup |
| `is_sales_tql` | ✅ Yes | Pre-acquisition enrichment |
| `tql_category` | ✅ Yes | Pre-acquisition enrichment |
| `initial_plan_type` | ✅ Yes | Set at first deal |
| `max_initial_default_hours` | ✅ Yes | Set at first deal |
| `max_initial_price_point` | ✅ Yes | Set at first deal |
| `first_week_mrr` | ⚠️ Review | Week 1 post-acquisition — technically post-acquisition but minimal; decision to include should be documented |
| `company_industry` | ✅ Yes | HubSpot enrichment at lead stage |
| `revenue_first_12_months` | 🎯 Target | This IS the target — not a feature |

> **Decision needed:** `first_week_mrr` is from the first billing week, not acquisition day. Decide whether to include it and document rationale here.

### 3.4 Data Quality Summary

> *(Fill after running cells above)*
> - How many rows are usable after accounting for nulls in the target?
> - What will you do with `company_industry` given ~40% missingness?
> - Any unexpected issues found?

---
## 4. Feature Distributions

Univariate analysis of each candidate predictor.

### 4.1 Categorical Features

In [ ]:
CATEGORICALS = [
    'channel',
    'broader_source',
    'service_offering',
    'is_sales_tql',
    'tql_category',
    'initial_plan_type',
    'max_initial_price_point',
    # 'company_industry'  # Uncomment after deciding missingness strategy
]

fig, axes = plt.subplots(len(CATEGORICALS), 1, figsize=(12, 4 * len(CATEGORICALS)))

for ax, col in zip(axes, CATEGORICALS):
    counts = df[col].value_counts(dropna=False)
    pct = counts / len(df) * 100
    labels = [f'{v} ({p:.1f}%)' for v, p in zip(counts.index.astype(str), pct)]
    sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax, color='steelblue')
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_ylabel('')
    for i, (count, p) in enumerate(zip(counts.values, pct)):
        ax.text(count + 1, i, f'{p:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 4.2 Numeric Features

In [ ]:
NUMERICS = [
    'max_initial_default_hours',
    'first_week_mrr',
]

fig, axes = plt.subplots(len(NUMERICS), 2, figsize=(14, 4 * len(NUMERICS)))

for i, col in enumerate(NUMERICS):
    # Histogram
    sns.histplot(df[col].dropna(), bins=40, ax=axes[i][0], color='steelblue')
    axes[i][0].set_title(f'{col} — Distribution', fontweight='bold')
    axes[i][0].set_xlabel(col)

    # Box plot
    sns.boxplot(x=df[col].dropna(), ax=axes[i][1], color='steelblue')
    axes[i][1].set_title(f'{col} — Box Plot', fontweight='bold')
    axes[i][1].set_xlabel(col)

plt.tight_layout()
plt.show()

### 4.3 Feature Distribution Observations

> *(Fill after running cells above)*
> - Are any categorical values too sparse to be useful (< 1% of rows)?
> - Are numeric features skewed? Will they need transformation before modeling?
> - Any unexpected values or anomalies?

---
## 5. Bivariate Analysis — Features vs. Target

This is the most important section for hypothesis generation. We want to understand whether each feature is associated with differences in CLV.

### 5.1 CLV by Service Offering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
order = df.groupby('service_offering')[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='service_offering', y=TARGET, order=order, ax=axes[0], palette='Blues')
axes[0].set_title('12-Month Revenue by Service Offering', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=30)

# Median bar
medians = df.groupby('service_offering')[TARGET].median().sort_values(ascending=False)
sns.barplot(x=medians.index, y=medians.values, ax=axes[1], palette='Blues_d')
axes[1].set_title('Median 12-Month Revenue by Service Offering', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Median Revenue (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Summary stats
print(df.groupby('service_offering')[TARGET].agg(['count', 'median', 'mean', 'std']).round(0))

### 5.2 CLV by Acquisition Channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order = df.groupby('channel')[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='channel', y=TARGET, order=order, ax=axes[0], palette='Blues')
axes[0].set_title('12-Month Revenue by Channel', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=30)

medians = df.groupby('channel')[TARGET].median().sort_values(ascending=False)
sns.barplot(x=medians.index, y=medians.values, ax=axes[1], palette='Blues_d')
axes[1].set_title('Median 12-Month Revenue by Channel', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Median Revenue (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(df.groupby('channel')[TARGET].agg(['count', 'median', 'mean', 'std']).round(0))

### 5.3 CLV by TQL Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# is_sales_tql (boolean)
sns.boxplot(data=df, x='is_sales_tql', y=TARGET, ax=axes[0], palette='Blues')
axes[0].set_title('12-Month Revenue by TQL Status', fontweight='bold')
axes[0].set_xlabel('Is TQL?')
axes[0].set_ylabel('Revenue (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# tql_category (granular)
order = df.groupby('tql_category')[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='tql_category', y=TARGET, order=order, ax=axes[1], palette='Blues')
axes[1].set_title('12-Month Revenue by TQL Category', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

print('TQL Status summary:')
print(df.groupby('is_sales_tql')[TARGET].agg(['count', 'median', 'mean']).round(0))
print('\nTQL Category summary:')
print(df.groupby('tql_category')[TARGET].agg(['count', 'median', 'mean']).sort_values('median', ascending=False).round(0))

### 5.4 CLV by Plan Type and Price Point

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order = df.groupby('initial_plan_type')[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='initial_plan_type', y=TARGET, order=order, ax=axes[0], palette='Blues')
axes[0].set_title('12-Month Revenue by Initial Plan Type', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=30)

order = df.groupby('max_initial_price_point')[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='max_initial_price_point', y=TARGET, order=order, ax=axes[1], palette='Blues')
axes[1].set_title('12-Month Revenue by Initial Price Point', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 5.5 CLV vs. Numeric Features (Scatter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, NUMERICS):
    ax.scatter(df[col], df[TARGET], alpha=0.3, color='steelblue', s=15)
    ax.set_title(f'{col} vs. 12-Month Revenue', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Revenue (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Add correlation annotation
    corr = df[[col, TARGET]].dropna().corr().iloc[0, 1]
    ax.annotate(f'r = {corr:.3f}', xy=(0.05, 0.93), xycoords='axes fraction', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

### 5.6 Bivariate Observations & Hypothesis Seeds

> *(Fill after running cells above. This section is the most important for Checkpoint 1.)*

Based on the bivariate analysis, the following hypotheses are proposed for the modeling phase:

1. **H1 (TQL):** TQL customers will have significantly higher predicted 12-month revenue than non-TQL customers, as TQL status captures company size and budget signals not otherwise visible at acquisition.

2. **H2 (Service offering):** *(Fill based on what you see in the data)*

3. **H3 (Channel):** *(Fill based on what you see in the data)*

4. **H4 (Plan type / price point):** *(Fill based on what you see in the data)*

5. **H5 (Starting hours / MRR):** *(Fill based on scatter plots)*

---
## 6. Segment Analysis

Cross-tabulations of key segment variables. Checks whether modeling cells are populated enough to be useful.

In [ ]:
# Channel × Service Offering — count heatmap
cross = pd.crosstab(df['service_offering'], df['channel'])

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Customer Count: Service Offering × Channel', fontweight='bold')
ax.set_xlabel('Channel')
ax.set_ylabel('Service Offering')
plt.tight_layout()
plt.show()

In [ ]:
# Channel × Service Offering — median revenue heatmap
cross_rev = df.groupby(['service_offering', 'channel'])[TARGET].median().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross_rev, annot=True, fmt='.0f', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Median 12-Month Revenue: Service Offering × Channel', fontweight='bold')
ax.set_xlabel('Channel')
ax.set_ylabel('Service Offering')
plt.tight_layout()
plt.show()

In [ ]:
# TQL × Service Offering
cross_tql = pd.crosstab(df['service_offering'], df['is_sales_tql'])
print('Count: Service Offering × TQL Status')
print(cross_tql)

cross_tql_rev = df.groupby(['service_offering', 'is_sales_tql'])[TARGET].median().unstack()
print('\nMedian Revenue: Service Offering × TQL Status')
print(cross_tql_rev.round(0))

### 6.1 Sparse Cell Check

> *(Fill after running cells above)*
> - Are there segment combinations with fewer than ~30 rows? These will be unreliable in modeling.
> - Will any categorical values need to be collapsed or treated as 'other'?

---
## 7. EDA Summary & Modeling Implications

This section is the EDA memo. It synthesizes findings into decisions that carry forward into preprocessing and modeling.

### 7.1 Dataset

| Item | Detail |
|---|---|
| Final row count (after quality filter) | *(fill)* |
| Date range | *(fill)* |
| Features retained | *(fill)* |
| Features excluded | *(fill and reason)* |

### 7.2 Target Variable

> *(Fill: distribution shape, whether log transform will be used, notable outliers)*

### 7.3 Key Findings

> *(Fill: 3–5 concrete findings from the bivariate analysis. E.g., "TQL customers have 2.4× the median revenue of non-TQL customers.")*

### 7.4 Preprocessing Decisions Motivated by EDA

| Decision | Rationale |
|---|---|
| Log-transform target | *(if applicable)* |
| `company_industry` handling | Treat nulls as separate category OR drop — decision: *(fill)* |
| Sparse categories | *(fill: which values will be collapsed)* |
| `first_week_mrr` inclusion | *(fill: include or exclude and why)* |

### 7.5 Hypotheses Carried into Modeling

> *(Copy from Section 5.6 once finalized)*

### 7.6 Data Risks

> *(Fill: any concerns that surfaced in EDA — unexpected distributions, sparse cells, cohort size, etc.)*